# Resume ODIR → mBRSET fine-tuning (EfficientNet-B0)

This notebook resumes the **existing binary `vascular_disease` fine-tuning checkpoint** from epoch 75 and continues through epoch 100. It restores the model, optimizer and scheduler states. It does **not** restart from ODIR weights or create a new train/validation/test split.

**Before using:** keep `resumed_last_checkpoint.pt` in `/home/jupyter-sandhyabr/`. Keep the original mBRSET training manifest (or its two source CSVs) and the mBRSET image files somewhere under that same directory. Use **Kernel → Restart & Run All** (or Run All) once the checks pass. The notebook stops with a clear message if required source data is missing.

In [6]:
# 1. Environment and paths
import os, random, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm

ROOT = Path("/home/jupyter-sandhyabr").resolve()
LAST_PATH = ROOT / "resumed_last_checkpoint.pt"
BEST_PATH = ROOT / "resumed_best_vascular_disease_model.pt"
HISTORY_PATH = ROOT / "resumed_training_history.csv"

IMAGE_SIZE = 512
BATCH_SIZE = 4
NUM_WORKERS = 0
TARGET_EPOCH = 100
SEED = 42

assert ROOT.is_dir(), f"Jupyter home folder not found: {ROOT}"
assert LAST_PATH.is_file(), f"Latest checkpoint not found: {LAST_PATH}"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("CUDA GPU is not available. In JupyterHub, select a GPU runtime/kernel before training.")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Root:", ROOT)
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Checkpoint:", LAST_PATH)


Root: /home/jupyter-sandhyabr
GPU: Tesla V100-PCIE-32GB
PyTorch: 2.5.1+cu121
Device: cuda
Checkpoint: /home/jupyter-sandhyabr/resumed_last_checkpoint.pt


In [10]:
import os

BASE = "/home/jupyter-sandhyabr"

old_name = os.path.join(
    BASE,
    "mbrset_vascular_disease_training_manifest - mbrset_vascular_disease_training_manifest.csv"
)

new_name = os.path.join(
    BASE,
    "mbrset_vascular_disease_training_manifest.csv"
)

if os.path.isfile(old_name):
    os.rename(old_name, new_name)
    print("File renamed successfully!")
else:
    print("Original file not found.")

print("Manifest found:", os.path.isfile(new_name))

File renamed successfully!
Manifest found: True


In [11]:
# 2. Find the ORIGINAL manifest/split and image files; do not make a new split.
# This searches only under ROOT and skips cache/version-control folders.
SKIP_DIRS = {".git", ".cache", ".ipynb_checkpoints", "__pycache__"}
manifest_candidates = []
rgb_manifest_candidates = []
label_candidates = []

for current, dirs, files in os.walk(ROOT):
    dirs[:] = [d for d in dirs if d not in SKIP_DIRS]
    lower = {f.lower(): f for f in files}
    if "mbrset_vascular_disease_training_manifest.csv" in lower:
        manifest_candidates.append(Path(current) / lower["mbrset_vascular_disease_training_manifest.csv"])
    if "mbrset_rgb_mask_manifest.csv" in lower:
        rgb_manifest_candidates.append(Path(current) / lower["mbrset_rgb_mask_manifest.csv"])
    if "labels_mbrset.csv" in lower:
        label_candidates.append(Path(current) / lower["labels_mbrset.csv"])

print("Training manifests:", manifest_candidates)
print("RGB-mask manifests:", rgb_manifest_candidates)
print("Label CSVs:", label_candidates)

if manifest_candidates:
    MANIFEST_PATH = manifest_candidates[0]
    df = pd.read_csv(MANIFEST_PATH)
    print("Using existing training manifest:", MANIFEST_PATH)
elif rgb_manifest_candidates and label_candidates:
    # Recreate the exact labelled manifest from the original patient-level split.
    rgb = pd.read_csv(rgb_manifest_candidates[0])
    labels = pd.read_csv(label_candidates[0])
    needed = {"file", "patient", "split", "image_path"}
    if not needed.issubset(rgb.columns):
        raise RuntimeError(f"RGB manifest is missing columns: {sorted(needed-set(rgb.columns))}")
    if not {"file", "patient", "vascular_disease"}.issubset(labels.columns):
        raise RuntimeError("labels_mbrset.csv must contain file, patient, and vascular_disease columns.")
    df = rgb.merge(labels[["file", "patient", "vascular_disease"]],
                   on=["file", "patient"], how="inner", validate="many_to_one")
    df = df[df["vascular_disease"].isin([0, 1])].copy()
    df["vascular_disease"] = df["vascular_disease"].astype(int)
    MANIFEST_PATH = ROOT / "mbrset_vascular_disease_training_manifest.csv"
    df.to_csv(MANIFEST_PATH, index=False)
    print("Rebuilt manifest using original split:", MANIFEST_PATH)
else:
    raise FileNotFoundError(
        "Could not find mBRSET training manifest or both source CSVs. "
        "Upload mbrset_vascular_disease_training_manifest.csv to /home/jupyter-sandhyabr, "
        "or upload mbrset_rgb_mask_manifest.csv and labels_mbrset.csv (preserving their original split)."
    )

required_cols = {"vascular_disease", "split", "patient"}
if not required_cols.issubset(df.columns):
    raise RuntimeError(f"Manifest missing required columns: {sorted(required_cols-set(df.columns))}")
if "file" not in df.columns and "filename" not in df.columns:
    raise RuntimeError("Manifest must have a 'file' or 'filename' column for image filenames.")

# Build a filename -> absolute path index. This repairs old Colab image_path values.
image_index = {}
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
for current, dirs, files in os.walk(ROOT):
    dirs[:] = [d for d in dirs if d not in SKIP_DIRS]
    for name in files:
        if Path(name).suffix.lower() in image_exts:
            image_index.setdefault(name, str(Path(current) / name))

filename_col = "file" if "file" in df.columns else "filename"
resolved_paths = []
missing_names = []
for _, row in df.iterrows():
    old_path = str(row.get("image_path", ""))
    filename = str(row[filename_col])
    if old_path and os.path.isfile(old_path):
        resolved_paths.append(old_path)
    elif filename in image_index:
        resolved_paths.append(image_index[filename])
    else:
        resolved_paths.append(None)
        missing_names.append(filename)

df["image_path"] = resolved_paths
if missing_names:
    raise FileNotFoundError(
        f"{len(missing_names)} manifest images were not found under {ROOT}. "
        f"Examples: {missing_names[:10]}. Upload/extract the complete mBRSET images under {ROOT}."
    )

df = df[df["vascular_disease"].isin([0, 1])].copy()
df["vascular_disease"] = df["vascular_disease"].astype(int)
if not set(df["split"].unique()).issubset({"train", "val", "test"}):
    raise RuntimeError("Unexpected split names. Expected train, val, test; refusing to create a new split.")
if set(df["split"].unique()) != {"train", "val", "test"}:
    raise RuntimeError("Manifest must contain train, val, and test splits.")
if (df.groupby("patient")["split"].nunique() > 1).any():
    raise RuntimeError("Patient leakage found across splits. Training cancelled.")

# Protect the known original split used by the epoch-75 checkpoint.
expected_counts = {"train": 2757, "val": 567, "test": 609}
actual_counts = df["split"].value_counts().to_dict()
if actual_counts != expected_counts:
    raise RuntimeError(
        f"Split counts do not match the checkpoint's original dataset. Expected {expected_counts}, got {actual_counts}. "
        "Do not train with a different split; restore the original manifest."
    )

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()
print("\nVerified original dataset split:")
print(df.groupby("split").agg(images=(filename_col, "size"), patients=("patient", "nunique")))
print("\nLabels by split:")
print(pd.crosstab(df["split"], df["vascular_disease"]))
print("\nAll image paths exist:", df["image_path"].map(os.path.isfile).all())


Training manifests: [PosixPath('/home/jupyter-sandhyabr/mbrset_vascular_disease_training_manifest.csv')]
RGB-mask manifests: []
Label CSVs: [PosixPath('/home/jupyter-sandhyabr/data/mbrset/labels_mbrset.csv')]
Using existing training manifest: /home/jupyter-sandhyabr/mbrset_vascular_disease_training_manifest.csv

Verified original dataset split:
       images  patients
split                  
test      609       191
train    2757       887
val       567       190

Labels by split:
vascular_disease     0    1
split                      
test               510   99
train             2276  481
val                472   95

All image paths exist: True


In [12]:
# 3. Dataset, transformations and data loaders (same training setup)
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

class VascularDiseaseDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row["image_path"]) as img:
            image = img.convert("RGB")
        image = self.transform(image)
        label = torch.tensor(float(row["vascular_disease"]), dtype=torch.float32)
        return image, label

train_loader = DataLoader(VascularDiseaseDataset(train_df, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(VascularDiseaseDataset(val_df, eval_transform),
                        batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(VascularDiseaseDataset(test_df, eval_transform),
                         batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

print("DataLoaders ready:")
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


DataLoaders ready:
Train batches: 690
Validation batches: 142
Test batches: 153


In [13]:
# 4. Recreate EfficientNet-B0 binary architecture and restore epoch-75 state
checkpoint = torch.load(LAST_PATH, map_location="cpu", weights_only=False)
required_ckpt_keys = {"epoch", "model_state", "optimizer_state", "scheduler_state",
                      "best_val_pr_auc", "best_epoch", "target", "image_size"}
missing_keys = required_ckpt_keys - set(checkpoint.keys())
if missing_keys:
    raise RuntimeError(f"Checkpoint is missing required keys: {sorted(missing_keys)}")
if checkpoint["target"] != "vascular_disease":
    raise RuntimeError(f"Wrong checkpoint target: {checkpoint['target']}")
if int(checkpoint["image_size"]) != IMAGE_SIZE:
    raise RuntimeError(f"Checkpoint image size is {checkpoint['image_size']}, expected {IMAGE_SIZE}")

saved_epoch = int(checkpoint["epoch"])
if saved_epoch >= TARGET_EPOCH:
    raise RuntimeError(f"Checkpoint is already at epoch {saved_epoch}; target is {TARGET_EPOCH}.")

# The checkpoint already contains the ODIR-initialized, mBRSET-fine-tuned model weights.
# We therefore recreate the binary head directly and restore model_state; ODIR checkpoint is not reloaded.
model = efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(1280, 1)
model = model.to(DEVICE)
model.load_state_dict(checkpoint["model_state"], strict=True)

num_positive = int((train_df["vascular_disease"] == 1).sum())
num_negative = int((train_df["vascular_disease"] == 0).sum())
if num_positive == 0:
    raise RuntimeError("Training split has no positive vascular_disease cases.")
pos_weight = torch.tensor([num_negative / num_positive], dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW([
    {"params": model.features.parameters(), "lr": 1e-5},
    {"params": model.classifier.parameters(), "lr": 1e-4},
], weight_decay=1e-4)
optimizer.load_state_dict(checkpoint["optimizer_state"])

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3)
scheduler.load_state_dict(checkpoint["scheduler_state"])

start_epoch = saved_epoch + 1
best_val_pr_auc = float(checkpoint["best_val_pr_auc"])
best_epoch = int(checkpoint["best_epoch"])

print("MODEL RESTORED SUCCESSFULLY")
print("Saved checkpoint epoch:", saved_epoch)
print("Next epoch:", start_epoch)
print("Best epoch:", best_epoch)
print("Best validation PR-AUC:", best_val_pr_auc)
print("Positive class weight:", pos_weight.item())


MODEL RESTORED SUCCESSFULLY
Saved checkpoint epoch: 75
Next epoch: 76
Best epoch: 66
Best validation PR-AUC: 0.2331273765095303
Positive class weight: 4.731808662414551


In [14]:
# 5. One-epoch train/validation function
from contextlib import nullcontext

def run_one_epoch(loader, training=True):
    model.train(training)
    total_loss = 0.0
    total_samples = 0
    all_labels, all_probs = [], []

    for images, labels in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True).float().view(-1, 1)
        if training:
            optimizer.zero_grad(set_to_none=True)

        grad_context = torch.enable_grad() if training else torch.no_grad()
        with grad_context:
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
                logits = model(images)
                loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()

        probs = torch.sigmoid(logits.detach())
        total_loss += float(loss.item()) * images.size(0)
        total_samples += images.size(0)
        all_labels.extend(labels.detach().cpu().numpy().ravel())
        all_probs.extend(probs.float().cpu().numpy().ravel())

    y_true = np.asarray(all_labels)
    y_prob = np.asarray(all_probs)
    roc_auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else float("nan")
    pr_auc = average_precision_score(y_true, y_prob)
    return total_loss / total_samples, float(roc_auc), float(pr_auc)

# Preserve previous epoch history, if present.
if HISTORY_PATH.is_file():
    history = pd.read_csv(HISTORY_PATH).to_dict("records")
else:
    history = []
history = [r for r in history if int(r.get("epoch", 0)) <= saved_epoch]

print("Ready to train from epoch", start_epoch, "through", TARGET_EPOCH)
print("Early stopping: DISABLED")


Ready to train from epoch 76 through 100
Early stopping: DISABLED


In [15]:
# 6. Continue training and save after EVERY completed epoch
# If the runtime disconnects, rerun this notebook; it will resume from LAST_PATH.
try:
    for epoch in range(start_epoch, TARGET_EPOCH + 1):
        train_loss, train_auc, train_ap = run_one_epoch(train_loader, training=True)
        val_loss, val_auc, val_ap = run_one_epoch(val_loader, training=False)
        scheduler.step(val_ap)

        row = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "train_roc_auc": float(train_auc),
            "train_pr_auc": float(train_ap),
            "val_loss": float(val_loss),
            "val_roc_auc": float(val_auc),
            "val_pr_auc": float(val_ap),
            "learning_rate_features": float(optimizer.param_groups[0]["lr"]),
            "learning_rate_classifier": float(optimizer.param_groups[1]["lr"]),
        }
        history = [r for r in history if int(r["epoch"]) != epoch]
        history.append(row)

        if val_ap > best_val_pr_auc:
            best_val_pr_auc = float(val_ap)
            best_epoch = epoch
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "best_val_pr_auc": best_val_pr_auc,
                "best_epoch": best_epoch,
                "target": "vascular_disease",
                "image_size": IMAGE_SIZE,
            }, BEST_PATH)

        # Atomic checkpoint write: avoids leaving a half-written latest checkpoint.
        temp_path = LAST_PATH.with_suffix(".tmp")
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "best_val_pr_auc": best_val_pr_auc,
            "best_epoch": best_epoch,
            "target": "vascular_disease",
            "image_size": IMAGE_SIZE,
        }, temp_path)
        os.replace(temp_path, LAST_PATH)
        pd.DataFrame(history).sort_values("epoch").to_csv(HISTORY_PATH, index=False)

        print(
            f"\nEpoch {epoch}/{TARGET_EPOCH} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
            f"Train ROC-AUC: {train_auc:.4f} | Val ROC-AUC: {val_auc:.4f} | "
            f"Train PR-AUC: {train_ap:.4f} | Val PR-AUC: {val_ap:.4f} | "
            f"Best PR-AUC: {best_val_pr_auc:.4f} (Epoch {best_epoch})"
        )

    print("\nTRAINING COMPLETED")
    print("Final saved epoch:", TARGET_EPOCH)
    print("Best epoch:", best_epoch)
    print("Best validation PR-AUC:", best_val_pr_auc)
    print("Latest checkpoint:", LAST_PATH)
    print("Best model:", BEST_PATH)
    print("History:", HISTORY_PATH)
except KeyboardInterrupt:
    print("\nTraining interrupted. Latest completed checkpoint:", LAST_PATH)
    print("Rerun this notebook to resume from the saved epoch.")
    raise


  0%|          | 0/690 [00:00<?, ?it/s]

  0%|          | 0/142 [00:00<?, ?it/s]


Epoch 76/100 | Train Loss: 1.6099 | Val Loss: 2.4539 | Train ROC-AUC: 0.8675 | Val ROC-AUC: 0.5806 | Train PR-AUC: 0.6740 | Val PR-AUC: 0.2267 | Best PR-AUC: 0.2331 (Epoch 66)


  0%|          | 0/690 [00:00<?, ?it/s]

  0%|          | 0/142 [00:00<?, ?it/s]


Epoch 77/100 | Train Loss: 1.5816 | Val Loss: 2.3461 | Train ROC-AUC: 0.8673 | Val ROC-AUC: 0.5645 | Train PR-AUC: 0.6798 | Val PR-AUC: 0.2227 | Best PR-AUC: 0.2331 (Epoch 66)


... (65 outputs left out)


Epoch 99/100 | Train Loss: 1.2407 | Val Loss: 2.6531 | Train ROC-AUC: 0.8754 | Val ROC-AUC: 0.5642 | Train PR-AUC: 0.6719 | Val PR-AUC: 0.2097 | Best PR-AUC: 0.2392 (Epoch 96)


  0%|          | 0/690 [00:00<?, ?it/s]

  0%|          | 0/142 [00:00<?, ?it/s]


Epoch 100/100 | Train Loss: 1.2580 | Val Loss: 2.2827 | Train ROC-AUC: 0.8694 | Val ROC-AUC: 0.5706 | Train PR-AUC: 0.6626 | Val PR-AUC: 0.2239 | Best PR-AUC: 0.2392 (Epoch 96)

TRAINING COMPLETED
Final saved epoch: 100
Best epoch: 96
Best validation PR-AUC: 0.23924312064262426
Latest checkpoint: /home/jupyter-sandhyabr/resumed_last_checkpoint.pt
Best model: /home/jupyter-sandhyabr/resumed_best_vascular_disease_model.pt
History: /home/jupyter-sandhyabr/resumed_training_history.csv


In [3]:

import os
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
from torchvision.models import efficientnet_b0
import torch.nn as nn

# Load the best validation model
BEST_PATH = "/home/jupyter-sandhyabr/resumed_best_vascular_disease_model.pt"

assert os.path.isfile(BEST_PATH), "Best model checkpoint not found."
assert "test_loader" in globals(), "Run the dataset and DataLoader setup cells first."

best_ckpt = torch.load(
    BEST_PATH,
    map_location="cpu",
    weights_only=False
)

# Recreate the model architecture
model = efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(1280, 1)
model.load_state_dict(best_ckpt["model_state"])
model = model.to(DEVICE)
model.eval()

print("Loaded best model from epoch:", best_ckpt["epoch"])
print("Best validation PR-AUC:", best_ckpt["best_val_pr_auc"])

# Test evaluation
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()

        all_probs.extend(probs)
        all_labels.extend(labels.numpy().ravel())

y_true = np.asarray(all_labels).astype(int)
y_prob = np.asarray(all_probs)

# Fixed threshold: 0.5
y_pred = (y_prob >= 0.5).astype(int)

# Metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_true, y_prob)
pr_auc = average_precision_score(y_true, y_prob)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

specificity = tn / (tn + fp) if (tn + fp) else 0

print("\n========== FINAL TEST RESULTS ==========")
print(f"Test images:       {len(y_true)}")
print(f"Accuracy:          {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision:         {precision:.4f}")
print(f"Recall/Sensitivity:{recall:.4f}")
print(f"Specificity:       {specificity:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"ROC-AUC:           {roc_auc:.4f}")
print(f"PR-AUC:            {pr_auc:.4f}")

print("\nConfusion Matrix:")
print(f"True Negatives:  {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives:  {tp}")

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=["No vascular disease", "Vascular disease"],
    zero_division=0
))


AssertionError: Run the dataset and DataLoader setup cells first.

In [2]:

import os
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
from torchvision.models import efficientnet_b0
import torch.nn as nn

# Load the best validation model
BEST_PATH = "/home/jupyter-sandhyabr/resumed_best_vascular_disease_model.pt"

assert os.path.isfile(BEST_PATH), "Best model checkpoint not found."
assert "test_loader" in globals(), "Run the dataset and DataLoader setup cells first."

best_ckpt = torch.load(
    BEST_PATH,
    map_location="cpu",
    weights_only=False
)

# Recreate the model architecture
model = efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(1280, 1)
model.load_state_dict(best_ckpt["model_state"])
model = model.to(DEVICE)
model.eval()

print("Loaded best model from epoch:", best_ckpt["epoch"])
print("Best validation PR-AUC:", best_ckpt["best_val_pr_auc"])

# Test evaluation
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()

        all_probs.extend(probs)
        all_labels.extend(labels.numpy().ravel())

y_true = np.asarray(all_labels).astype(int)
y_prob = np.asarray(all_probs)

# Fixed threshold: 0.5
y_pred = (y_prob >= 0.5).astype(int)

# Metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_true, y_prob)
pr_auc = average_precision_score(y_true, y_prob)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

specificity = tn / (tn + fp) if (tn + fp) else 0

print("\n========== FINAL TEST RESULTS ==========")
print(f"Test images:       {len(y_true)}")
print(f"Accuracy:          {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision:         {precision:.4f}")
print(f"Recall/Sensitivity:{recall:.4f}")
print(f"Specificity:       {specificity:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"ROC-AUC:           {roc_auc:.4f}")
print(f"PR-AUC:            {pr_auc:.4f}")

print("\nConfusion Matrix:")
print(f"True Negatives:  {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives:  {tp}")

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=["No vascular disease", "Vascular disease"],
    zero_division=0
))


AssertionError: Run the dataset and DataLoader setup cells first.

In [4]:
print("Test loader:", "test_loader" in globals())

print("Test images:", len(test_loader.dataset))
print("Test batches:", len(test_loader))

print("Test split:", len(test_df))

Test loader: False


NameError: name 'test_loader' is not defined

## After training

The latest checkpoint is `resumed_last_checkpoint.pt` in `/home/jupyter-sandhyabr/`. The best validation model is `resumed_best_vascular_disease_model.pt`, and metrics are in `resumed_training_history.csv`.

This notebook does not evaluate the test set during training, does not alter the patient split, and does not load the separate ODIR checkpoint again. The epoch-75 checkpoint already contains the ODIR-initialized model after mBRSET fine-tuning.

In [5]:
print("test_loader exists:", "test_loader" in globals())
print("test_df exists:", "test_df" in globals())

test_loader exists: False
test_df exists: False


In [6]:
import os

BASE = "/home/jupyter-sandhyabr"

print("Manifest:", os.path.isfile(
    BASE + "/mbrset_vascular_disease_training_manifest.csv"
))

print("Image directory:", os.path.isdir(
    BASE + "/data/mbrset/images"
))

print("Checkpoint:", os.path.isfile(
    BASE + "/resumed_best_vascular_disease_model.pt"
))

Manifest: True
Image directory: True
Checkpoint: True


In [7]:

import os
import pandas as pd
import torch
import numpy as np

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

BASE = "/home/jupyter-sandhyabr"

MANIFEST_PATH = os.path.join(
    BASE, "mbrset_vascular_disease_training_manifest.csv"
)

IMAGE_DIR = os.path.join(BASE, "data", "mbrset", "images")

IMAGE_SIZE = 512
BATCH_SIZE = 4
NUM_WORKERS = 0

# Load original manifest
df = pd.read_csv(MANIFEST_PATH)

# Restore image paths
df["image_path"] = df["file"].astype(str).apply(
    lambda x: os.path.join(IMAGE_DIR, x)
)

# Verify original split and labels
assert set(df["split"].unique()) == {"train", "val", "test"}
assert df.groupby("patient")["split"].nunique().max() == 1
assert df.groupby("split").size().to_dict() == {
    "train": 2757, "val": 567, "test": 609
}
assert df["vascular_disease"].isin([0, 1]).all()
assert df["image_path"].map(os.path.isfile).all()

# Separate original test set
test_df = df[df["split"] == "test"].copy()

# Evaluation transformation
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Dataset
class VascularTestDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        with Image.open(row["image_path"]) as img:
            image = img.convert("RGB")
            image = eval_transform(image)

        label = torch.tensor(
            float(row["vascular_disease"]),
            dtype=torch.float32
        )

        return image, label

# Create test loader
test_dataset = VascularTestDataset(test_df)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

print("TEST DATALOADER CREATED")
print("Test images:", len(test_dataset))
print("Test batches:", len(test_loader))
print("Test patients:", test_df["patient"].nunique())
print("All image paths exist:", test_df["image_path"].map(os.path.isfile).all())


TEST DATALOADER CREATED
Test images: 609
Test batches: 153
Test patients: 191
All image paths exist: True


In [8]:

import torch
import numpy as np
from torchvision.models import efficientnet_b0
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BEST_PATH = "/home/jupyter-sandhyabr/resumed_best_vascular_disease_model.pt"

# Load best checkpoint
checkpoint = torch.load(
    BEST_PATH,
    map_location="cpu",
    weights_only=False
)

# Rebuild EfficientNet-B0
model = efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(1280, 1)

model.load_state_dict(checkpoint["model_state"])
model = model.to(DEVICE)
model.eval()

print("Loaded best model from epoch:", checkpoint["epoch"])
print("Best validation PR-AUC:", checkpoint["best_val_pr_auc"])

# Generate test predictions
y_true = []
y_prob = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        logits = model(images)
        probabilities = torch.sigmoid(logits).cpu().numpy().ravel()

        y_prob.extend(probabilities)
        y_true.extend(labels.numpy().ravel())

y_true = np.asarray(y_true).astype(int)
y_prob = np.asarray(y_prob)

# Fixed threshold of 0.5
y_pred = (y_prob >= 0.5).astype(int)

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

roc_auc = roc_auc_score(y_true, y_prob)
pr_auc = average_precision_score(y_true, y_prob)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

specificity = tn / (tn + fp) if (tn + fp) else 0

print("\n========== FINAL TEST RESULTS ==========")
print("Test images:", len(y_true))
print(f"Accuracy:    {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"F1-score:    {f1:.4f}")
print(f"ROC-AUC:     {roc_auc:.4f}")
print(f"PR-AUC:      {pr_auc:.4f}")

print("\nConfusion Matrix:")
print("TN:", tn, "| FP:", fp)
print("FN:", fn, "| TP:", tp)

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=["No vascular disease", "Vascular disease"],
    zero_division=0
))


Loaded best model from epoch: 96
Best validation PR-AUC: 0.23924312064262426

========== FINAL TEST RESULTS ==========
Test images: 609
Accuracy:    0.7734 (77.34%)
Precision:   0.1321
Recall:      0.0707
Specificity: 0.9098
F1-score:    0.0921
ROC-AUC:     0.4774
PR-AUC:      0.1520

Confusion Matrix:
TN: 464 | FP: 46
FN: 92 | TP: 7

Classification Report:
                     precision    recall  f1-score   support

No vascular disease       0.83      0.91      0.87       510
   Vascular disease       0.13      0.07      0.09        99

           accuracy                           0.77       609
          macro avg       0.48      0.49      0.48       609
       weighted avg       0.72      0.77      0.74       609



In [16]:
import torch
import numpy as np

model.eval()

all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)

        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy().reshape(-1)

        all_probs.extend(probs.tolist())
        all_labels.extend(labels.cpu().numpy().reshape(-1).tolist())

y_true = np.asarray(all_labels, dtype=int)
y_prob = np.asarray(all_probs, dtype=float)

print("Predictions recreated successfully")
print("y_true type:", type(y_true))
print("y_prob type:", type(y_prob))
print("Test images:", len(y_true))

Predictions recreated successfully
y_true type: <class 'numpy.ndarray'>
y_prob type: <class 'numpy.ndarray'>
Test images: 609


In [15]:
print("Positive cases:", int(np.sum(y_true == 1)))
print("Negative cases:", int(np.sum(y_true == 0)))

print("\nPrediction probabilities:")
print("Minimum:", y_prob.min())
print("Maximum:", y_prob.max())
print("Mean:", y_prob.mean())
print("Median:", np.median(y_prob))

print("\nActual positive cases:")
print("Mean probability:", y_prob[y_true == 1].mean())

print("\nActual negative cases:")
print("Mean probability:", y_prob[y_true == 0].mean())

print("\nProbability distribution:")
for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    print(
        f"Threshold {threshold:.1f}:",
        int(np.sum(y_prob >= threshold)),
        "predicted positive"
    )

Positive cases: 99
Negative cases: 510

Prediction probabilities:
Minimum: 1.389024419040652e-05
Maximum: 0.8258642554283142
Mean: 0.15430929718298003
Median: 0.07383643090724945

Actual positive cases:
Mean probability: 0.13771644881060896

Actual negative cases:
Mean probability: 0.15753026186702854

Probability distribution:
Threshold 0.1: 259 predicted positive
Threshold 0.2: 169 predicted positive
Threshold 0.3: 115 predicted positive
Threshold 0.4: 72 predicted positive
Threshold 0.5: 53 predicted positive
Threshold 0.6: 31 predicted positive
Threshold 0.7: 11 predicted positive
Threshold 0.8: 2 predicted positive
Threshold 0.9: 0 predicted positive


In [17]:
from sklearn.metrics import roc_auc_score, average_precision_score

print("ROC-AUC:", roc_auc_score(y_true, y_prob))
print("PR-AUC:", average_precision_score(y_true, y_prob))

print("\nChecking reversed predictions:")
print("Reversed ROC-AUC:", roc_auc_score(y_true, 1 - y_prob))
print("Reversed PR-AUC:", average_precision_score(y_true, 1 - y_prob))

print("\nProbability comparison:")
print("Positive median:", np.median(y_prob[y_true == 1]))
print("Negative median:", np.median(y_prob[y_true == 0]))

ROC-AUC: 0.47740146563675967
PR-AUC: 0.15199550521214544

Checking reversed predictions:
Reversed ROC-AUC: 0.5225985343632402
Reversed PR-AUC: 0.20570999084998623

Probability comparison:
Positive median: 0.08496560156345367
Negative median: 0.07172137498855591


In [18]:
import pandas as pd

manifest = pd.read_csv(
    "/home/jupyter-sandhyabr/mbrset_vascular_disease_training_manifest.csv"
)

print("Columns:")
print(manifest.columns.tolist())

print("\nLabel distribution:")
print(manifest["vascular_disease"].value_counts(dropna=False))

print("\nLabel distribution by split:")
print(
    pd.crosstab(
        manifest["split"],
        manifest["vascular_disease"],
        margins=True
    )
)

print("\nSample records:")
print(
    manifest[
        ["file", "patient", "split", "vascular_disease"]
    ].head(10)
)

Columns:
['file', 'patient', 'split', 'CVD_related', 'HT_target', 'image_path', 'mask_path', 'vascular_disease']

Label distribution:
vascular_disease
0    3258
1     675
Name: count, dtype: int64

Label distribution by split:
vascular_disease     0    1   All
split                            
test               510   99   609
train             2276  481  2757
val                472   95   567
All               3258  675  3933

Sample records:
         file  patient  split  vascular_disease
0     1.1.jpg        1   test                 0
1     1.2.jpg        1   test                 0
2     1.4.jpg        1   test                 0
3    10.1.jpg       10  train                 0
4    10.3.jpg       10  train                 0
5    10.4.jpg       10  train                 0
6   100.2.jpg      100  train                 0
7   100.3.jpg      100  train                 0
8  1000.1.jpg     1000  train                 0
9  1000.2.jpg     1000  train                 0


In [2]:
import pandas as pd

manifest = pd.read_csv(
    "/home/jupyter-sandhyabr/mbrset_vascular_disease_training_manifest.csv"
)

print("Manifest loaded successfully")
print("Total images:", len(manifest))
print("Columns:", manifest.columns.tolist())

Manifest loaded successfully
Total images: 3933
Columns: ['file', 'patient', 'split', 'CVD_related', 'HT_target', 'image_path', 'mask_path', 'vascular_disease']


In [3]:
print("CVD_related values:")
print(manifest["CVD_related"].value_counts(dropna=False))

print("\nComparison of CVD_related and vascular_disease:")
print(
    pd.crosstab(
        manifest["CVD_related"],
        manifest["vascular_disease"],
        margins=True,
        dropna=False
    )
)

print("\nMissing values:")
print(manifest[["CVD_related", "vascular_disease"]].isna().sum())

print("\nUnique values:")
print("CVD_related:", manifest["CVD_related"].unique())
print("vascular_disease:", manifest["vascular_disease"].unique())

CVD_related values:
CVD_related
0    3080
1     853
Name: count, dtype: int64

Comparison of CVD_related and vascular_disease:
vascular_disease     0    1   All
CVD_related                      
0                 3080    0  3080
1                  178  675   853
All               3258  675  3933

Missing values:
CVD_related         0
vascular_disease    0
dtype: int64

Unique values:
CVD_related: [1 0]
vascular_disease: [0 1]
